In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls
!pip install -e ".[experiment]"

In [ ]:
!pip install -e .

Obtaining file:///content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... canceled
ERROR: Operation cancelled by user
^C


In [ ]:
!pip -q install -U pip

# core deps
!pip -q install torch transformers datasets evaluate accelerate sentencepiece einops tqdm

# if your repo uses peft/wandb/syne-tune (optional; only needed for finetuning)
!pip -q install peft wandb

# install your project in editable mode (adjust extras if you have them)
!pip -q install -e .

In [2]:
import slicegpt
from slicegpt import rotate, model_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)

slicegpt imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/model_utils.py


In [ ]:
from slicegpt.model_adapter import ModelAdapter
from slicegpt import hf_utils
# IMPORTANT: ensure the adapter class is registered
import slicegpt.adapters.t5_adapter  # noqa: F401

MODEL_ID = "google/flan-t5-base"

# this should return (model_adapter, tokenizer) using your new adapter
model_adapter, tokenizer = hf_utils.get_model_and_tokenizer(MODEL_ID, model_path=None, token=None)

print("Adapter:", type(model_adapter).__name__)
print("HF model:", type(model_adapter.model).__name__)
print("Hidden size:", model_adapter.hidden_size)
print("Seq len:", model_adapter.seqlen)

# check seq2seq hooks exist
print("has encoder layers:", hasattr(model_adapter, "get_encoder_layers"))
print("has decoder layers:", hasattr(model_adapter, "get_decoder_layers"))

In [ ]:
'''
Expected:
Adapter is T5ModelAdapter
has encoder layers and has decoder layers are True
If adapter is wrong: you likely need to ensure t5_adapter.py is imported somewhere at package init (e.g. slicegpt/adapters/__init__.py).
'''

In [ ]:
import torch
from slicegpt import hf_utils

MODEL_ID = "google/flan-t5-base"
model_adapter, tok = hf_utils.get_model_and_tokenizer(MODEL_ID, model_path=None, token=None)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model_adapter.model.to(device)
model.eval()

print("device:", device)
print("adapter.use_cache:", getattr(model_adapter, "use_cache", None))
print("model.config.use_cache:", getattr(model.config, "use_cache", None))

# ---- 1) basic seq2seq forward (teacher forcing) ----
src = "translate English to German: The house is wonderful."
tgt = "Das Haus ist wunderbar."

inputs = tok(src, return_tensors="pt", truncation=True, max_length=64).to(device)

with tok.as_target_tokenizer():
    labels = tok(tgt, return_tensors="pt", truncation=True, max_length=64).input_ids.to(device)

out = model(**inputs, labels=labels)
print("forward OK | loss:", float(out.loss), "| logits:", tuple(out.logits.shape))

# ---- 2) generate path ----
gen_ids = model.generate(**inputs, max_new_tokens=32)
print("generate OK |", tok.decode(gen_ids[0], skip_special_tokens=True))

# ---- 3) explicit encoder then decoder (forces cross-attention code path) ----
enc_out = model.get_encoder()(
    input_ids=inputs["input_ids"],
    attention_mask=inputs.get("attention_mask", None),
    return_dict=True,
)

decoder_start = model.config.decoder_start_token_id
decoder_input_ids = torch.tensor([[decoder_start]], device=device)

dec_out = model.get_decoder()(
    input_ids=decoder_input_ids,
    encoder_hidden_states=enc_out.last_hidden_state,
    encoder_attention_mask=inputs.get("attention_mask", None),
    return_dict=True,
)

print("enc/dec OK | enc:", tuple(enc_out.last_hidden_state.shape), "| dec:", tuple(dec_out.last_hidden_state.shape))

# ---- 4) check layer lists (used by rotate_and_slice_seq2seq) ----
enc_layers = model_adapter.get_encoder_layers()
dec_layers = model_adapter.get_decoder_layers()
print("num encoder layers:", len(enc_layers))
print("num decoder layers:", len(dec_layers))


In [ ]:
import slicegpt.rotate as rotate
print(rotate.__file__)


In [ ]:

import torch
from slicegpt import hf_utils, data_utils, rotate, layernorm_fusion
from slicegpt.slicing_scheduler import ConstSlicingScheduler
from slicegpt.config import config

# ---- config ----
MODEL_ID = "google/flan-t5-base"
CAL_DATASET = "squad"
NSAMPLES = 8
BATCH_SIZE = 2
MAX_SEQLEN = 128
SPARSITY = 0.20
ROUND_INTERVAL = 8
FINAL_ORIENTATION = "random"

config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.dtype = torch.float16

# ---- load ----
model_adapter, tok = hf_utils.get_model_and_tokenizer(MODEL_ID, model_path=None, token=None, dtype=config.dtype)
model = model_adapter.model
model.to(config.device)
model.eval()
model_adapter.use_cache = False

print("loaded:", type(model_adapter).__name__, "| hidden:", model_adapter.hidden_size, "| seqlen:", model.seqlen)

# ---- IMPORTANT: make layernorms compatible with get_signals() ----
layernorm_fusion.replace_layers(model_adapter)
layernorm_fusion.fuse_modules(model_adapter)

# ---- tiny calibration loader ----
dataset = data_utils.get_dataset(CAL_DATASET)
train_loader = data_utils.prepare_dataloader(
    dataset=dataset["train"],
    tokenizer=tok,
    max_seqlen=MAX_SEQLEN,
    batch_size=BATCH_SIZE,
    nsamples=NSAMPLES,
    varied_seqlen=False,
    seed=42,
)

# ---- scheduler ----
new_dim = int((1 - SPARSITY) * model_adapter.hidden_size)
new_dim -= new_dim % ROUND_INTERVAL
print("target dim:", new_dim)

scheduler = ConstSlicingScheduler(new_dim)

# ---- rotate + slice seq2seq ----
torch.cuda.empty_cache()
rotate.rotate_and_slice_seq2seq(
    model_adapter,
    train_loader,
    scheduler,
    final_orientation=FINAL_ORIENTATION,
)

print("rotate_and_slice_seq2seq: DONE")
print("slicing_conf:", model_adapter.slicing_conf)


In [ ]:
type(model_adapter.model.encoder.block[0]), type(model_adapter.model.decoder.block[0])


In [ ]:
#now that finally it works, we can run run_slicegpt!!!

In [2]:
import os, textwrap

# Where to save logs and models in your Drive
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs_flant5")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_flant5")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

#sparsities = [0.0, 0.1, 0.25, 0.4, 0.6]
sparsities = [0.1]
datasets = ["squad"]

print("Logs in:", LOG_DIR)
print("Models in:", MODEL_DIR)

Logs in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_flant5
Models in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5


In [3]:
# Cell 1: Slicing with correct dtype
import os
import subprocess
from datetime import datetime

MODEL_ID = "google/flan-t5-base"

def run_slicegpt(dataset, sparsity):
    log_name = f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}.txt".replace(".", "p")
    log_path = os.path.join(LOG_DIR, log_name)

    if os.path.exists(log_path):
        print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
        return

    save_dir = os.path.join(MODEL_DIR, f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", MODEL_ID,
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--dtype", "fp32",  # ← CRITICAL FIX: Use fp32 for T5
        "--cal-batch-size", "8"
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            f.write(line)

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py --model google/flan-t5-base --cal-dataset squad --save-dir /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10 --sparsity 0.1 --device cuda:0 --no-wandb --dtype fp32 --cal-batch-size 8
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_flant5/google-flan-t5-base_squad_s0p10ptxt
Start: 2026-01-05 19:11:40.081504

Running SliceGPT experiment.
PyTorch device: cuda:0
Number of available cuda devices: 1
Loading google/flan-t5-base config and model weights from Hugging Face
Loading model done
Loading dataset: squad
Loading dataset done
Preparing dataloader
Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors
Preparing dataloader done
Preparing test dataloader
Preparing test dataloader done
Repla

In [ ]:
#now let's produce some evaluation

In [4]:
!pip -q uninstall -y peft
!pip -q install "transformers==4.41.0" "accelerate>=0.28,<1.0" "peft==0.12.0"


In [5]:
!python -c "import transformers, peft, accelerate; print(transformers.__version__, peft.__version__, accelerate.__version__)"


4.41.0 0.12.0 0.34.2


In [6]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
import torch
from slicegpt.config import config

config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.dtype = torch.float32   # add the missing field


In [ ]:
# Cell 2: Evaluation function (fixed dtype)
import os, json, logging
import torch
import lm_eval
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM

from slicegpt import hf_utils
from slicegpt.config import config

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def eval(args):
    logger.info("Running Evaluation")

    logger.info(
        f"Loading sliced {args['model']} from {args['sliced_model_path']} "
        f"with sparsity {args['sparsity']}"
    )

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        args["model"],
        args["sliced_model_path"],
        sparsity=args["sparsity"],
        token=None,
        round_interval=args.get("round_interval", None),
    )

    # Disable tie_weights if it tries to re-tie (can break sliced models)
    if hasattr(model_adapter.model, "tie_weights"):
        model_adapter.model.tie_weights = lambda *x, **kw: None

    # ← CRITICAL FIX: Use fp32 to match slicing dtype
    dtype = torch.float32
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_adapter.model.to(device=device, dtype=dtype)
    model_adapter.model.eval()

    # Verify model is sane before evaluation
    logger.info("Running sanity checks...")
    has_nan = False
    for name, param in model_adapter.model.named_parameters():
        if not torch.isfinite(param).all():
            logger.error(f"✗ Non-finite values in {name}")
            has_nan = True
            break

    if has_nan:
        raise RuntimeError("Model has NaN/Inf parameters - cannot evaluate!")

    logger.info("✓ All parameters are finite")

    hflm = HFLM(
        pretrained=model_adapter.model,
        tokenizer=tokenizer,
        batch_size=args["batch_size"],
        max_length=int(args.get("max_input_len", 512) or 512),
    )

    # Tasks
    if args["tasks"] is None:
        logger.warning("args['tasks'] is None -> using ALL_TASKS (can be slow).")
        task_names = tasks.ALL_TASKS
    else:
        if isinstance(args["tasks"], str):
            patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
        else:
            patterns = args["tasks"]
        task_names = lm_eval_utils.pattern_match(patterns, ALL_TASKS)

    logger.info(f"Selected Tasks: {task_names}")

    gen_kwargs = None
    if args.get("max_gen_toks") is not None:
        gen_kwargs = f"max_gen_toks={args['max_gen_toks']},do_sample=False"

    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=False,
        log_samples=False,
        gen_kwargs=gen_kwargs,
    )

    results = results["results"]

    os.makedirs(args["save_dir"], exist_ok=True)
    sparsity_tag = f"{args['sparsity']:.2f}"
    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}.json",
    )

    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {result_path}")

    # Print key metrics
    logger.info("\n" + "="*60)
    logger.info("EVALUATION RESULTS")
    logger.info("="*60)
    for task, metrics in results.items():
        logger.info(f"\nTask: {task}")
        for metric, value in metrics.items():
            if not metric.endswith("_stderr"):
                logger.info(f"  {metric}: {value}")
    logger.info("="*60)

    return results

In [7]:
# Modified Cell 2: Evaluation function with projection layer support for slicing just the encoder

import os, json, logging
import torch
import lm_eval
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM

from slicegpt import hf_utils
from slicegpt.config import config

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def eval(args):
    logger.info("Running Evaluation")

    logger.info(
        f"Loading sliced {args['model']} from {args['sliced_model_path']} "
        f"with sparsity {args['sparsity']}"
    )

    # UNCHANGED: Load model (projection layer added automatically in hf_utils.load_sliced_model)
    model_adapter, tokenizer = hf_utils.load_sliced_model(
        args["model"],
        args["sliced_model_path"],
        sparsity=args["sparsity"],
        token=None,
        round_interval=args.get("round_interval", None),
    )

    # Disable tie_weights if it tries to re-tie (can break sliced models)
    if hasattr(model_adapter.model, "tie_weights"):
        model_adapter.model.tie_weights = lambda *x, **kw: None

    # Use fp32 to match slicing dtype
    dtype = torch.float32
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_adapter.model.to(device=device, dtype=dtype)
    model_adapter.model.eval()

    # NEW: Move projection layer to device if it exists
    if hasattr(model_adapter.model, 'encoder_projection'):
        model_adapter.model.encoder_projection.to(device=device, dtype=dtype)
        logger.info("✓ Projection layer moved to device")

    # Verify model is sane before evaluation
    logger.info("Running sanity checks...")
    has_nan = False
    for name, param in model_adapter.model.named_parameters():
        if not torch.isfinite(param).all():
            logger.error(f"✗ Non-finite values in {name}")
            has_nan = True
            break

    if has_nan:
        raise RuntimeError("Model has NaN/Inf parameters - cannot evaluate!")

    logger.info("✓ All parameters are finite")

    # NEW: Test forward pass to verify model works
    logger.info("Testing forward pass...")
    try:
        test_input = tokenizer(
            "translate English to French: Hello world",
            return_tensors="pt"
        )
        test_input = {k: v.to(device) for k, v in test_input.items()}

        with torch.no_grad():
            test_output = model_adapter.model.generate(
                **test_input,
                max_length=20,
                num_beams=1
            )

        decoded = tokenizer.decode(test_output[0], skip_special_tokens=True)
        logger.info(f"✓ Forward pass successful!")
        logger.info(f"  Test output: {decoded}")
    except Exception as e:
        logger.error(f"✗ Forward pass failed: {e}")
        import traceback
        traceback.print_exc()
        raise

    # UNCHANGED: Create HFLM for evaluation
    hflm = HFLM(
        pretrained=model_adapter.model,
        tokenizer=tokenizer,
        batch_size=args["batch_size"],
        max_length=int(args.get("max_input_len", 512) or 512),
    )

    # Tasks
    if args["tasks"] is None:
        logger.warning("args['tasks'] is None -> using ALL_TASKS (can be slow).")
        task_names = tasks.ALL_TASKS
    else:
        if isinstance(args["tasks"], str):
            patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
        else:
            patterns = args["tasks"]
        task_names = lm_eval_utils.pattern_match(patterns, ALL_TASKS)

    logger.info(f"Selected Tasks: {task_names}")

    gen_kwargs = None
    if args.get("max_gen_toks") is not None:
        gen_kwargs = f"max_gen_toks={args['max_gen_toks']},do_sample=False"

    # UNCHANGED: Run evaluation
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=False,
        log_samples=False,
        gen_kwargs=gen_kwargs,
    )

    results = results["results"]

    os.makedirs(args["save_dir"], exist_ok=True)
    sparsity_tag = f"{args['sparsity']:.2f}"
    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}.json",
    )

    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {result_path}")

    # Print key metrics
    logger.info("\n" + "="*60)
    logger.info("EVALUATION RESULTS")
    logger.info("="*60)
    for task, metrics in results.items():
        logger.info(f"\nTask: {task}")
        for metric, value in metrics.items():
            if not metric.endswith("_stderr"):
                logger.info(f"  {metric}: {value}")
    logger.info("="*60)

    return results


INFO - NumExpr defaulting to 8 threads.
INFO - PyTorch version 2.9.0+cu126 available.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# Cell 3: Run evaluation
model = "google/flan-t5-base"
model_name = model.replace("/", "-")
datasets = ['squad']
sparsities = [0.10]

dir_prefix = "squadEval"
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_flant5")

results = None

for dataset in datasets:
    for sparsity in sparsities:
        EVAL_DIR = os.path.join(
            BASE_EVAL_DIR,
            dir_prefix,
            f"{model_name}_{dataset}"
        )
        os.makedirs(EVAL_DIR, exist_ok=True)

        args = {
            "model": model,
            "sliced_model_path": os.path.join(MODEL_DIR, f"{model_name}_{dataset}_s{sparsity:.2f}".replace(".", "p")),
            "sparsity": sparsity,
            "save_dir": EVAL_DIR,
            "tasks": ["squadv2"],
            "num_fewshot": 0,
            "batch_size": 8,
            "round_interval": 8,
            "limit": 100,  # ← Changed from 10 (still small but more representative)
            "max_input_len": 512,
            "max_gen_toks": 16,  # ← Added explicit generation limit
        }

        print(f"\n{'='*60}")
        print(f"Evaluating: {model_name} on {dataset} with sparsity {sparsity}")
        print(f"{'='*60}\n")

        results = eval(args)


Evaluating: google-flan-t5-base on squad with sparsity 0.1

INFO - Running Evaluation
INFO - Loading sliced google/flan-t5-base from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10 with sparsity 0.1
INFO - Loading google/flan-t5-base config  from Hugging Face


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Detected encoder-only sliced checkpoint (decoder at 768 dims)
INFO - Applying encoder-only slicing to model skeleton...
INFO - ✓ Encoder-only slicing applied: encoder 688 dims, decoder 768 dims
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10
INFO - Detected encoder output dimension: 688 from model skeleton
INFO - Cross-attention K/V will be sliced to encoder output dim: 688
INFO - ======================================================================
INFO - ENCODER-ONLY SLICING DETECTED
INFO - ======================================================================
INFO - Embedding dimension: 768
INFO - Encoder input dimension: 688
INFO - Adding projection layer: 768 → 688
INFO - ✓ Replaced encoder.embed_tokens with projected version
INFO - ✓

100%|██████████| 100/100 [02:33<00:00,  1.54s/it]

INFO - Running loglikelihood requests



100%|██████████| 100/100 [00:02<00:00, 49.87it/s]
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/squadv2/task.py:40: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  squad_metric = datasets.load_metric("squad_v2")
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for 

INFO - Saved results to /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squadEval/google-flan-t5-base_squad/results_s0.10_squadv2.json
INFO - 
INFO - EVALUATION RESULTS
INFO - ============================================================
INFO - 
Task: squadv2
INFO -   exact,none: 55.0
INFO -   exact_stderr,none: N/A
INFO -   f1,none: 55.0
INFO -   f1_stderr,none: N/A
INFO -   HasAns_exact,none: 0.0
INFO -   HasAns_exact_stderr,none: N/A
INFO -   HasAns_f1,none: 0.0
INFO -   HasAns_f1_stderr,none: N/A
INFO -   NoAns_exact,none: 100.0
INFO -   NoAns_exact_stderr,none: N/A
INFO -   NoAns_f1,none: 100.0
INFO -   NoAns_f1_stderr,none: N/A
INFO -   best_exact,none: 55.0
INFO -   best_exact_stderr,none: N/A
INFO -   best_f1,none: 55.0
INFO -   best_f1_stderr,none: N/A
INFO -   alias: squadv2
INFO - ============================================================


/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [8]:
# Cell 4: Quick diagnostic test (run BEFORE evaluation)
import torch
from slicegpt import hf_utils

print("Running diagnostic test...")

model_adapter, tokenizer = hf_utils.load_sliced_model(
    "google/flan-t5-base",
    "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10",
    sparsity=0.10
)

model = model_adapter.model.to(device="cuda", dtype=torch.float32).eval()

# Check 1: Weight tying
print("\n" + "="*60)
print("CHECK 1: Weight Tying")
print("="*60)
tied = model.lm_head.weight.data_ptr() == model.shared.weight.data_ptr()
print(f"✓ Weights tied: {tied}" if tied else f"✗ Weights NOT tied: {tied}")
print(f"  Shared shape: {model.shared.weight.shape}")
print(f"  LM head shape: {model.lm_head.weight.shape}")

# Check 2: NaN/Inf
print("\n" + "="*60)
print("CHECK 2: Parameter Sanity")
print("="*60)
has_nan = False
for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        print(f"✗ Non-finite values in {name}")
        has_nan = True
        break
if not has_nan:
    print("✓ All parameters are finite")

# Check 3: Forward pass
print("\n" + "="*60)
print("CHECK 3: Forward Pass")
print("="*60)
prompt = "translate English to French: Hello"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    try:
        out = model(**inputs, labels=inputs.input_ids)
        loss_val = out.loss.item()
        logits_finite = torch.isfinite(out.logits).all().item()

        print(f"✓ Loss: {loss_val:.4f}" if loss_val < 100 else f"✗ Loss: {loss_val:.4f} (too high!)")
        print(f"✓ Logits finite: {logits_finite}" if logits_finite else f"✗ Logits have NaN/Inf")
    except Exception as e:
        print(f"✗ Forward pass failed: {e}")

# Check 4: Generation
print("\n" + "="*60)
print("CHECK 4: Generation Quality")
print("="*60)
test_prompts = [
    "translate English to French: Hello",
    "translate English to German: Thank you",
    "Answer: What is 2+2?"
]

all_good = True
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=16, num_beams=1)
        text = tokenizer.decode(gen[0], skip_special_tokens=True)

    # Check if it's garbage
    is_garbage = "activitati" in text.lower() or len(set(text.split())) < 2
    status = "✗ GARBAGE" if is_garbage else "✓ OK"

    print(f"\n{status}")
    print(f"  Prompt: {prompt}")
    print(f"  Output: {repr(text)}")

    if is_garbage:
        all_good = False

print("\n" + "="*60)
if all_good and not has_nan and tied:
    print("✓✓✓ ALL CHECKS PASSED - Model appears healthy!")
else:
    print("✗✗✗ CHECKS FAILED - Model is broken, do NOT evaluate yet")
print("="*60)

Running diagnostic test...
INFO - Loading google/flan-t5-base config  from Hugging Face


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Detected encoder-only sliced checkpoint (decoder at 768 dims)
INFO - Applying encoder-only slicing to model skeleton...
INFO - ✓ Encoder-only slicing applied: encoder 688 dims, decoder 768 dims
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10
INFO - Detected encoder output dimension: 688 from model skeleton
INFO - Cross-attention K/V will be sliced to encoder output dim: 688
INFO - ======================================================================
INFO - ENCODER-ONLY SLICING DETECTED
INFO - ======================================================================
INFO - Embedding dimension: 768
INFO - Encoder input dimension: 688
INFO - Adding projection layer: 768 → 688
INFO - ✓ Replaced encoder.embed_tokens with projected version
INFO - ✓

In [8]:
# Cell 4: Quick diagnostic test (run BEFORE evaluation)
# This version handles encoder-only sliced models with projection layer

import torch
from slicegpt import hf_utils

print("="*70)
print("DIAGNOSTIC TEST FOR ENCODER-ONLY SLICED T5")
print("="*70)

# Configuration
model_name = "google/flan-t5-base"
checkpoint_path = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10"
sparsity = 0.10

print(f"\nModel: {model_name}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Sparsity: {sparsity}")

try:
    print("\n" + "="*70)
    print("STEP 1: Loading Model")
    print("="*70)

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        model_name,
        checkpoint_path,
        sparsity=sparsity,
        round_interval=8
    )

    print("✓ Model loaded successfully")

    model = model_adapter.model

    # Check dimensions
    print("\n" + "="*70)
    print("STEP 2: Dimension Check")
    print("="*70)

    # Get actual dimensions
    embedding_dim = model.shared.weight.shape[1]
    enc_dim = model.encoder.block[0].layer[0].SelfAttention.q.weight.shape[1]
    dec_dim = model.decoder.block[0].layer[0].SelfAttention.q.weight.shape[1]

    print(f"Embedding dimension: {embedding_dim}")
    print(f"Encoder input dimension: {enc_dim}")
    print(f"Decoder input dimension: {dec_dim}")

    # Check if encoder-only slicing
    is_encoder_only = (enc_dim < embedding_dim) and (dec_dim == embedding_dim)

    if is_encoder_only:
        print(f"✓ Encoder-only slicing detected correctly")
        print(f"  Encoder: {embedding_dim} → {enc_dim} (sliced)")
        print(f"  Decoder: {dec_dim} (unchanged)")
    else:
        print(f"✗ Unexpected dimensions!")
        print(f"  Expected: encoder < embedding == decoder")
        print(f"  Got: encoder={enc_dim}, embedding={embedding_dim}, decoder={dec_dim}")

    # Check 3: Projection layer
    print("\n" + "="*70)
    print("STEP 3: Projection Layer Check")
    print("="*70)

    has_projection = hasattr(model, 'encoder_projection')
    print(f"{'✓' if has_projection else '✗'} Projection layer exists: {has_projection}")

    if has_projection:
        proj_shape = model.encoder_projection.weight.shape
        print(f"  Projection shape: {proj_shape}")
        expected_shape = (enc_dim, embedding_dim)
        if proj_shape == expected_shape:
            print(f"  ✓ Projection shape correct: {proj_shape}")
        else:
            print(f"  ✗ Projection shape wrong: expected {expected_shape}, got {proj_shape}")
    else:
        print(f"  ✗ WARNING: No projection layer - model will fail!")

    # Check 4: Weight tying
    print("\n" + "="*70)
    print("STEP 4: Weight Tying")
    print("="*70)

    tied = model.lm_head.weight.data_ptr() == model.shared.weight.data_ptr()
    print(f"{'✓' if tied else '✗'} Weights tied: {tied}")
    print(f"  Shared shape: {model.shared.weight.shape}")
    print(f"  LM head shape: {model.lm_head.weight.shape}")

    if not tied:
        print("  ✗ WARNING: Weights should be tied!")

    # Check 5: Cross-attention dimensions
    print("\n" + "="*70)
    print("STEP 5: Decoder Cross-Attention Check")
    print("="*70)

    # Check first decoder layer cross-attention
    dec_layer = model.decoder.block[0].layer[1].EncDecAttention
    cross_k_dim = dec_layer.k.weight.shape[1]
    cross_v_dim = dec_layer.v.weight.shape[1]
    cross_q_dim = dec_layer.q.weight.shape[1]

    print(f"Cross-attention K input: {cross_k_dim}")
    print(f"Cross-attention V input: {cross_v_dim}")
    print(f"Cross-attention Q input: {cross_q_dim}")

    # K and V should match encoder output (688), Q should match decoder (768)
    cross_kv_correct = (cross_k_dim == enc_dim) and (cross_v_dim == enc_dim)
    cross_q_correct = (cross_q_dim == dec_dim)

    if cross_kv_correct and cross_q_correct:
        print(f"✓ Cross-attention dimensions correct")
        print(f"  K/V accept encoder output: {enc_dim} dims")
        print(f"  Q processes decoder hidden: {dec_dim} dims")
    else:
        print(f"✗ Cross-attention dimensions incorrect!")
        print(f"  Expected: K/V={enc_dim}, Q={dec_dim}")
        print(f"  Got: K={cross_k_dim}, V={cross_v_dim}, Q={cross_q_dim}")

    # Check 6: Parameter sanity
    print("\n" + "="*70)
    print("STEP 6: Parameter Sanity")
    print("="*70)

    has_nan = False
    nan_params = []
    for name, param in model.named_parameters():
        if not torch.isfinite(param).all():
            has_nan = True
            nan_params.append(name)
            if len(nan_params) <= 5:
                print(f"✗ Non-finite values in {name}")

    if has_nan:
        print(f"✗ Found {len(nan_params)} parameters with NaN/Inf")
        if len(nan_params) > 5:
            print(f"  (showing first 5, total: {len(nan_params)})")
    else:
        print("✓ All parameters are finite")

    # Check 7: Move to GPU and test forward pass
    print("\n" + "="*70)
    print("STEP 7: GPU Movement and Forward Pass")
    print("="*70)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32

    print(f"Moving to device: {device}, dtype: {dtype}")
    model.to(device=device, dtype=dtype)
    model.eval()

    # Move projection layer too
    if has_projection:
        model.encoder_projection.to(device=device, dtype=dtype)
        print("✓ Projection layer moved to device")

    print("\nTesting forward pass...")
    prompt = "translate English to French: Hello world"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        try:
            outputs = model(**inputs, labels=inputs.input_ids)
            loss_val = outputs.loss.item()
            logits_finite = torch.isfinite(outputs.logits).all().item()

            if loss_val < 100 and logits_finite:
                print(f"✓ Forward pass successful!")
                print(f"  Loss: {loss_val:.4f}")
                print(f"  Logits finite: {logits_finite}")
            else:
                print(f"✗ Forward pass returned bad values!")
                print(f"  Loss: {loss_val:.4f} {'(too high!)' if loss_val >= 100 else ''}")
                print(f"  Logits finite: {logits_finite}")
        except Exception as e:
            print(f"✗ Forward pass failed!")
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
            raise

    # Check 8: Generation quality
    print("\n" + "="*70)
    print("STEP 8: Generation Quality")
    print("="*70)

    test_prompts = [
        ("translate English to French: Hello", "Bonjour"),
        ("translate English to German: Thank you", "Danke"),
        ("translate English to Spanish: Good morning", "Buenos días"),
    ]

    all_good = True
    for prompt, expected_word in test_prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            try:
                gen = model.generate(**inputs, max_new_tokens=16, num_beams=1)
                text = tokenizer.decode(gen[0], skip_special_tokens=True)

                # Check if output is reasonable
                is_garbage = (
                    len(text.strip()) == 0 or  # Empty
                    len(set(text.split())) < 2 or  # Too repetitive
                    "activitati" in text.lower() or  # Known garbage pattern
                    len(text) > 100  # Too long
                )

                status = "✗" if is_garbage else "✓"
                print(f"\n{status} Prompt: {prompt}")
                print(f"  Output: {repr(text)}")

                if is_garbage:
                    all_good = False
                    print(f"  WARNING: Output looks like garbage!")

            except Exception as e:
                print(f"\n✗ Prompt: {prompt}")
                print(f"  Generation failed: {e}")
                all_good = False

    # Final summary
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)

    checks_passed = []
    checks_failed = []

    if is_encoder_only:
        checks_passed.append("Encoder-only slicing detected")
    else:
        checks_failed.append("Incorrect dimensions")

    if has_projection:
        checks_passed.append("Projection layer exists")
    else:
        checks_failed.append("No projection layer")

    if tied:
        checks_passed.append("Weights properly tied")
    else:
        checks_failed.append("Weights not tied")

    if cross_kv_correct and cross_q_correct:
        checks_passed.append("Cross-attention configured correctly")
    else:
        checks_failed.append("Cross-attention misconfigured")

    if not has_nan:
        checks_passed.append("No NaN/Inf parameters")
    else:
        checks_failed.append(f"{len(nan_params)} parameters with NaN/Inf")

    if all_good:
        checks_passed.append("Generation quality good")
    else:
        checks_failed.append("Generation quality poor")

    print("\nPassed:")
    for check in checks_passed:
        print(f"  ✓ {check}")

    if checks_failed:
        print("\nFailed:")
        for check in checks_failed:
            print(f"  ✗ {check}")

    print("\n" + "="*70)

    if len(checks_failed) == 0:
        print("✓✓✓ ALL CHECKS PASSED - Model is healthy!")
        print("You can proceed with evaluation.")
    elif len(checks_failed) <= 2 and not has_nan and has_projection:
        print("⚠️  SOME CHECKS FAILED - Model may work but not optimally")
        print("You can try evaluation but monitor results carefully.")
    else:
        print("✗✗✗ CRITICAL CHECKS FAILED - Model is broken!")
        print("DO NOT evaluate yet. Fix the issues above first.")

    print("="*70)

except Exception as e:
    print("\n" + "="*70)
    print("✗✗✗ DIAGNOSTIC TEST CRASHED")
    print("="*70)
    print(f"Error: {e}")
    print("\nFull traceback:")
    import traceback
    traceback.print_exc()
    print("\nThe model is broken - DO NOT proceed with evaluation!")
    print("="*70)

DIAGNOSTIC TEST FOR ENCODER-ONLY SLICED T5

Model: google/flan-t5-base
Checkpoint: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10
Sparsity: 0.1

STEP 1: Loading Model
INFO - Loading google/flan-t5-base config  from Hugging Face


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Detected encoder-only sliced checkpoint (decoder at 768 dims)
INFO - Applying encoder-only slicing to model skeleton...
INFO - ✓ Encoder-only slicing applied: encoder 688 dims, decoder 768 dims
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10
INFO - Detected encoder output dimension: 688 from model skeleton
INFO - Cross-attention K/V will be sliced to encoder output dim: 688
INFO - ======================================================================
INFO - ENCODER-ONLY SLICING DETECTED
INFO - ======================================================================
INFO - Embedding dimension: 768
INFO - Encoder input dimension: 688
INFO - Adding projection layer: 768 → 688
INFO - ✓ Projection layer created and encoder forward method wrapped
I

In [ ]:
import slicegpt, slicegpt.rotate, slicegpt.hf_utils
from slicegpt.adapters import t5_adapter

print("slicegpt:", slicegpt.__file__)
print("rotate.py:", slicegpt.rotate.__file__)
print("hf_utils.py:", slicegpt.hf_utils.__file__)
print("t5_adapter.py:", t5_adapter.__file__)


In [ ]:
'''
Extra: your eval code rewrite (clean + robust)
Here’s a clean final version of your eval() that:
doesn’t assume config.dtype exists (fixes the 'dtype' KeyError)
allows max_input_len to suppress the 649>512 warning (by passing gen kwargs to lm-eval)
saves results exactly where you expect
'''

In [ ]:
import torch
from datasets import load_dataset
from slicegpt import hf_utils

MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5/google-flan-t5-base_squad_s0p10"

model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL, SLICED_PATH, sparsity=0.10, token=None, round_interval=8
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_adapter.model.to(device=device, dtype=torch.float16).eval()   # <<< FIX

ds = load_dataset("rajpurkar/squad_v2", split="validation[:3]")

for i, ex in enumerate(ds):
    prompt = (
        "Answer the question based on the context.\n"
        f"Context: {ex['context']}\n"
        f"Question: {ex['question']}\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        out = model_adapter.model.generate(**inputs, max_new_tokens=32, do_sample=False)

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print("\n---", i, "---")
    print("GEN:", repr(text))


In [ ]:
import torch
from datasets import load_dataset
from slicegpt import hf_utils

MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5_base/google-flan-t5-base_squad_s0p10"

model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL, SLICED_PATH, sparsity=0.10, token=None, round_interval=8
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_adapter.model.to(device=device, dtype=torch.float16).eval()

print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("eos_token_id:", model.config.eos_token_id)
print("pad_token_id:", model.config.pad_token_id)

ds = load_dataset("rajpurkar/squad_v2", split="validation[:1]")

ex = ds[0]
prompt = (
    "Answer the question based on the context.\n"
    f"Context: {ex['context']}\n"
    f"Question: {ex['question']}\n"
    "Answer:"
)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)

ids = out[0].tolist()
print("GEN IDS (first 20):", ids[:20])
print("GEN TOKENS (first 20):", tokenizer.convert_ids_to_tokens(ids[:20]))
print("DECODED:", repr(tokenizer.decode(out[0], skip_special_tokens=False)))
print("DECODED (skip special):", repr(tokenizer.decode(out[0], skip_special_tokens=True)))


In [ ]:
# ============================
# Next-step debugging script:
# - verifies weight tying
# - checks for NaNs/Infs
# - inspects top-10 next-token logits
# - tests generation with/without forbidding <pad>
# - (optionally) re-ties weights correctly
# ============================

import torch
from datasets import load_dataset
from slicegpt import hf_utils

# ---- config ----
MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5_base/google-flan-t5-base_squad_s0p10"
SPARSITY = 0.10
ROUND_INTERVAL = 8
DTYPE = torch.float16            # use float32 if you want more stability while debugging
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 32

# ---- load sliced model ----
model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL,
    SLICED_PATH,
    sparsity=SPARSITY,
    token=None,
    round_interval=ROUND_INTERVAL,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_adapter.model.to(device=device, dtype=DTYPE).eval()

print("\n=== CONFIG TOKENS ===")
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("eos_token_id:", model.config.eos_token_id)
print("pad_token_id:", model.config.pad_token_id)

# ---- check shared + lm_head shapes/dtypes + tying ----
print("\n=== WEIGHT TIE CHECK ===")
print("shared.weight:", tuple(model.shared.weight.shape), model.shared.weight.dtype)
print("lm_head.weight:", tuple(model.lm_head.weight.shape), model.lm_head.weight.dtype)
try:
    tied = (model.shared.weight.data_ptr() == model.lm_head.weight.data_ptr())
except Exception as e:
    tied = False
    print("Could not compare data_ptr:", e)
print("tied(shared <-> lm_head):", tied)

# ---- OPTIONAL: try re-tying weights (do this especially if tied == False) ----
# IMPORTANT: In your eval script you were overriding tie_weights with a no-op.
# Here we actually call tie_weights to restore tying if the architecture expects it.
print("\n=== TRY CALLING tie_weights() ===")
if hasattr(model, "tie_weights") and callable(getattr(model, "tie_weights")):
    try:
        model.tie_weights()
        tied2 = (model.shared.weight.data_ptr() == model.lm_head.weight.data_ptr())
        print("tie_weights() called. tied now:", tied2)
    except Exception as e:
        print("tie_weights() failed:", repr(e))
else:
    print("Model has no callable tie_weights().")

# ---- get 1 example ----
ds = load_dataset("rajpurkar/squad_v2", split="validation[:1]")
ex = ds[0]
prompt = (
    "Answer the question based on the context.\n"
    f"Context: {ex['context']}\n"
    f"Question: {ex['question']}\n"
    "Answer:"
)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)

print("\n=== PROMPT TAIL ===")
print(repr(prompt[-200:]))

# ---- encoder hidden state sanity (NaN/Inf check) ----
print("\n=== ENCODER HIDDEN STATS ===")
with torch.no_grad():
    enc_out = model.get_encoder()(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        return_dict=True,
    )
    hs = enc_out.last_hidden_state
    print("shape:", tuple(hs.shape), "dtype:", hs.dtype)
    print("nan?:", torch.isnan(hs).any().item(), "inf?:", torch.isinf(hs).any().item())
    print("mean:", hs.float().mean().item(), "std:", hs.float().std().item())

# ---- one-step decoder logits inspection (find why <pad> dominates) ----
print("\n=== ONE-STEP NEXT-TOKEN LOGITS (TOP-10) ===")
with torch.no_grad():
    start_id = torch.tensor([[model.config.decoder_start_token_id]], device=device)
    out1 = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        decoder_input_ids=start_id,
        return_dict=True,
    )
    logits = out1.logits[0, -1]  # vocab logits at first generated position
    print("logits nan?:", torch.isnan(logits).any().item(), "inf?:", torch.isinf(logits).any().item())

    topv, topi = torch.topk(logits, 10)
    topi_list = topi.tolist()
    top_tokens = tokenizer.convert_ids_to_tokens(topi_list)
    print("TOP-10 (id, token, logit):")
    for tid, tok, val in zip(topi_list, top_tokens, topv.float().tolist()):
        print(f"  {tid:6d}  {tok:>12s}  {val: .6f}")

    # also report pad logit rank-ish
    pad_id = tokenizer.pad_token_id
    pad_logit = logits[pad_id].float().item()
    print("\nPAD token id:", pad_id, "pad_logit:", pad_logit)

# ---- generation test 1: plain generate ----
print("\n=== GENERATE (PLAIN) ===")
with torch.no_grad():
    gen_plain = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

print("GEN IDS (first 30):", gen_plain[0][:30].tolist())
print("GEN TOKENS (first 30):", tokenizer.convert_ids_to_tokens(gen_plain[0][:30].tolist()))
print("DECODED (skip special):", repr(tokenizer.decode(gen_plain[0], skip_special_tokens=True)))
print("DECODED (with special):", repr(tokenizer.decode(gen_plain[0], skip_special_tokens=False)))

# ---- generation test 2: forbid <pad> so you can see if model can produce *any* non-special token ----
print("\n=== GENERATE (FORBID <pad>) ===")
with torch.no_grad():
    gen_no_pad = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        bad_words_ids=[[tokenizer.pad_token_id]],  # forbid pad
    )

print("GEN IDS (first 30):", gen_no_pad[0][:30].tolist())
print("GEN TOKENS (first 30):", tokenizer.convert_ids_to_tokens(gen_no_pad[0][:30].tolist()))
print("DECODED (skip special):", repr(tokenizer.decode(gen_no_pad[0], skip_special_tokens=True)))
print("DECODED (with special):", repr(tokenizer.decode(gen_no_pad[0], skip_special_tokens=False)))

# ---- Optional extra: also forbid <pad> and force eos early to see if eos is ever plausible ----
print("\n=== OPTIONAL: GENERATE (FORBID <pad>, SMALL max_new_tokens=8) ===")
with torch.no_grad():
    gen_no_pad_short = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        bad_words_ids=[[tokenizer.pad_token_id]],
    )

print("GEN TOKENS:", tokenizer.convert_ids_to_tokens(gen_no_pad_short[0].tolist()))
print("DECODED (skip special):", repr(tokenizer.decode(gen_no_pad_short[0], skip_special_tokens=True)))

# ============================
# What to paste back to me:
# 1) tied(shared <-> lm_head): True/False  (before and after tie_weights())
# 2) encoder hidden nan?/inf? + mean/std
# 3) TOP-10 next-token logits (id/token/logit)
# 4) DECODED results for plain + forbid <pad>
# ============================


In [ ]:
import torch
from datasets import load_dataset
from slicegpt import hf_utils

MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5_base/google-flan-t5-base_squad_s0p10"
SPARSITY = 0.10
ROUND_INTERVAL = 8
MAX_INPUT_LEN = 512

def any_nan(t: torch.Tensor) -> bool:
    return torch.isnan(t).any().item()

def any_inf(t: torch.Tensor) -> bool:
    return torch.isinf(t).any().item()

# ----------------------------
# 1) Load sliced model
# ----------------------------
model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL, SLICED_PATH, sparsity=SPARSITY, token=None, round_interval=ROUND_INTERVAL
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# IMPORTANT: first run in float32 to avoid fp16 overflow confusion
model = model_adapter.model.to(device=device, dtype=torch.float32).eval()

print("\n=== CONFIG ===")
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("eos_token_id:", model.config.eos_token_id)
print("pad_token_id:", model.config.pad_token_id)

print("\n=== SHAPES ===")
print("shared:", tuple(model.shared.weight.shape), model.shared.weight.dtype)
print("lm_head:", tuple(model.lm_head.weight.shape), model.lm_head.weight.dtype)

# Hard fail if dims inconsistent (your case right now)
shared_dim = model.shared.weight.shape[1]
lm_head_dim = model.lm_head.weight.shape[1]
if shared_dim != lm_head_dim:
    print("\n!!! ERROR: shared_dim != lm_head_dim !!!")
    print("This sliced checkpoint is inconsistent: shared is sliced but lm_head is not.")
    print("You must fix slicing (slice/rotate lm_head too) and re-save the model.")
    # keep going anyway to localize NaNs, but do NOT trust eval numbers.

# ----------------------------
# 2) Check if any PARAMETERS are NaN/Inf
# ----------------------------
print("\n=== PARAM NaN/Inf CHECK (first hits) ===")
nan_hits = 0
inf_hits = 0
for name, p in model.named_parameters():
    if any_nan(p):
        print("NaN PARAM:", name, tuple(p.shape), p.dtype)
        nan_hits += 1
        if nan_hits >= 10:
            break
for name, p in model.named_parameters():
    if any_inf(p):
        print("Inf PARAM:", name, tuple(p.shape), p.dtype)
        inf_hits += 1
        if inf_hits >= 10:
            break
if nan_hits == 0 and inf_hits == 0:
    print("No NaN/Inf found in parameters (good).")

# ----------------------------
# 3) Build one SQuADv2 prompt
# ----------------------------
ds = load_dataset("rajpurkar/squad_v2", split="validation[:1]")
ex = ds[0]
prompt = (
    "Answer the question based on the context.\n"
    f"Context: {ex['context']}\n"
    f"Question: {ex['question']}\n"
    "Answer:"
)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)

print("\n=== PROMPT TAIL ===")
print(repr(prompt[-200:]))

# ----------------------------
# 4) Encoder hidden stats (you said these are OK)
# ----------------------------
print("\n=== ENCODER OUTPUT STATS ===")
with torch.no_grad():
    enc = model.get_encoder()(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        return_dict=True,
    )
    hs = enc.last_hidden_state
    print("enc.last_hidden_state:", tuple(hs.shape), hs.dtype)
    print("nan?", any_nan(hs), "inf?", any_inf(hs), "mean", hs.mean().item(), "std", hs.std().item())

# ----------------------------
# 5) Decoder hidden states + logits stats
#    This tells you if NaNs appear BEFORE lm_head or only in logits.
# ----------------------------
print("\n=== DECODER + LOGITS STATS (1 step) ===")
with torch.no_grad():
    start_id = torch.tensor([[model.config.decoder_start_token_id]], device=device)

    out = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        decoder_input_ids=start_id,
        output_hidden_states=True,
        return_dict=True,
    )

    # decoder last hidden
    dec_last = out.decoder_hidden_states[-1][0, -1]  # (d_model,)
    print("decoder_last_hidden:", tuple(dec_last.shape), dec_last.dtype)
    print("decoder_last nan?", any_nan(dec_last), "inf?", any_inf(dec_last),
          "min", dec_last.min().item(), "max", dec_last.max().item())

    # logits at first generated position
    logits = out.logits[0, -1]  # (vocab,)
    print("logits:", tuple(logits.shape), logits.dtype)
    print("logits nan?", any_nan(logits), "inf?", any_inf(logits),
          "min", logits.min().item(), "max", logits.max().item())

    # top-10 tokens (only meaningful if logits are not NaN)
    if not any_nan(logits):
        topv, topi = torch.topk(logits, 10)
        toks = tokenizer.convert_ids_to_tokens(topi.tolist())
        print("\nTOP-10 (id, token, logit):")
        for tid, tok, val in zip(topi.tolist(), toks, topv.tolist()):
            print(f"  {tid:6d}  {tok:>12s}  {val: .6f}")
    else:
        print("\nLogits are NaN, so decoding will collapse (explains <pad> loop).")

# ----------------------------
# 6) Try generate in float32 (should still fail if NaNs)
# ----------------------------
print("\n=== GENERATE (float32) ===")
with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=32, do_sample=False)

print("GEN TOKENS (first 20):", tokenizer.convert_ids_to_tokens(gen[0][:20].tolist()))
print("DECODED (skip special):", repr(tokenizer.decode(gen[0], skip_special_tokens=True)))
print("DECODED (with special):", repr(tokenizer.decode(gen[0], skip_special_tokens=False)))


In [ ]:
#to test
import torch
from datasets import load_dataset
from slicegpt import hf_utils

MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5_base/google-flan-t5-base_squad_s0p10"

# 1) Load sliced model
model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL,
    SLICED_PATH,
    sparsity=0.10,
    token=None,
    round_interval=8,
)

# 2) Use float32 for debugging first (later you can switch to float16)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_adapter.model.to(device=device, dtype=torch.float32).eval()

# 3) Quick param finite check (fail fast)
bad = []
for n, p in model_adapter.model.named_parameters():
    if p is not None and not torch.isfinite(p).all():
        bad.append((n, tuple(p.shape), p.dtype))
if bad:
    print("FOUND NON-FINITE PARAMS:")
    for x in bad[:30]:
        print("  ", x)
    raise RuntimeError("Checkpoint has NaN/Inf params. Re-slice after fixing rotate.py.")
else:
    print("All parameters are finite ✅")

# 4) Load 3 examples
ds = load_dataset("rajpurkar/squad_v2", split="validation[:3]")

# 5) Generate
for i, ex in enumerate(ds):
    prompt = (
        "Answer the question based on the context.\n"
        f"Context: {ex['context']}\n"
        f"Question: {ex['question']}\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        out = model_adapter.model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            num_beams=1,
        )

    gen_ids = out[0].tolist()
    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)

    print("\n---", i, "---")
    print("Q:", ex["question"])
    print("GEN IDS (first 20):", gen_ids[:20])
    print("GEN TEXT:", repr(gen_text))

    # extra: detect degenerate <pad> loop
    if len(gen_ids) > 5 and all(t == tokenizer.pad_token_id for t in gen_ids[:10]):
        print("⚠️ Looks like a <pad> loop. That usually means logits are broken (often NaNs).")


In [ ]:
import torch
from datasets import load_dataset
from slicegpt import hf_utils

MODEL = "google/flan-t5-base"
SLICED_PATH = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_flant5_base/google-flan-t5-base_squad_s0p10_FIXED"

model_adapter, tokenizer = hf_utils.load_sliced_model(
    MODEL,
    SLICED_PATH,
    sparsity=0.10,
    round_interval=8,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_adapter.model.to(device=device, dtype=torch.float32).eval()

ds = load_dataset("rajpurkar/squad_v2", split="validation[:3]")

for i, ex in enumerate(ds):
    prompt = (
        "Answer the question based on the context.\n"
        f"Context: {ex['context']}\n"
        f"Question: {ex['question']}\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        out = model_adapter.model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
        )

    print("\n---", i, "---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
model = model_adapter.model

print("shared:", model.shared.weight.shape)
print("lm_head:", model.lm_head.weight.shape)
print("tied:", model.shared.weight.data_ptr() == model.lm_head.weight.data_ptr())

for n, p in model.named_parameters():
    if not torch.isfinite(p).all():
        raise RuntimeError(f"NaN found in {n}")


In [ ]:
shared: (32128, D)
lm_head: (32128, D)
tied: True
